In [40]:
vocab = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3
}


In [41]:
from collections import Counter, defaultdict

def get_stats(vocab):
    """Count frequency of adjacent symbol pairs"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs


def merge_vocab(pair, vocab):
    """Merge the most frequent pair"""
    merged_vocab = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)

    for word, freq in vocab.items():
        new_word = word.replace(bigram, replacement)
        merged_vocab[new_word] = freq

    return merged_vocab


def train_bpe(vocab, num_merges):
    """Learn BPE merges"""
    # add end-of-word symbol and split chars
    vocab = {
        " ".join(list(word)) + " </w>": freq
        for word, freq in vocab.items()
    }

    merges = []

    for _ in range(num_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break

        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

    return merges


In [42]:
def bpe_tokenize(word, merges):
    tokens = list(word) + ["</w>"]

    for merge in merges:
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == merge[0] and tokens[i+1] == merge[1]:
                tokens[i:i+2] = ["".join(merge)]
            else:
                i += 1

    return tokens


In [43]:
vocab = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3
}

merges = train_bpe(vocab, num_merges=10)

print("Learned merges:")
for m in merges:
    print(m)

print("\nTokenization examples:")
print(bpe_tokenize("lower", merges))
print(bpe_tokenize("newest", merges))


Learned merges:
('e', 's')
('es', 't')
('est', '</w>')
('l', 'o')
('lo', 'w')
('n', 'e')
('ne', 'w')
('new', 'est</w>')
('low', '</w>')
('w', 'i')

Tokenization examples:
['low', 'e', 'r', '</w>']
['newest</w>']
